# Week 3: Entity Extraction

Goal: pull structured facts out of cleaned remarks, including bedrooms, bathrooms, square footage, price, and amenities.

1. The labeled dataset
2. How the extractor works
3. Scores on the dev and test sets
4. Error analysis
5. Amenity check
6. Comparison with the database fields

Only short phrases from remarks are shown so the notebook is safe to commit.

In [1]:
import math
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from scripts.entity_extractor import EntityExtractor
from scripts.evaluate_entities import load_labeled, score

DATA = ROOT / "data/processed"
extractor = EntityExtractor()

## 1. Labeled dataset

250 cleaned remarks were labeled by hand (rules in `docs/entity_labeling_guidelines.md`).
- **Dev (100 remarks):** used while writing the extraction rules
- **Test (150 remarks):** held out and only scored once at the end

In [2]:
labels = pd.read_csv(DATA / "entity_labels.csv")
counts = labels.pivot_table(index="label", columns="split", values="value", aggfunc="count")
counts.loc["total"] = counts.sum()
counts[["dev", "test"]]

split,dev,test
label,,
bathrooms,58,91
bedrooms,67,98
price,1,2
sqft,38,47
total,164,238


Price is rare because most agents don't repeat the asking price in the remarks.

## 2. How the extractor works

Regex rules find counts like "3 spacious bedrooms", then skip ones that aren't the whole home ("2 additional bedrooms", "ADU with 1 bedroom", "6500 square feet lot"). Amenities come from the Week 1 taxonomy.

In [3]:
example = ("Charming 3 bedroom, 2 full bathrooms and 1 half bath home with 1850 square feet "
           "on a 7000 square feet lot. Offered at $899000. ADU with 1 bedroom. "
           "Features a pool, quartz countertops, and a 2-car garage.")
extractor.extract_all(example)

{'bedrooms': 3,
 'bathrooms': 2.5,
 'price': 899000,
 'sqft': 1850,
 'amenities': ['quartz countertops', 'pool', 'garage']}

## 3. Scores

A prediction only counts if the number is exactly right. The target is 85%+ F1.

In [4]:
dev_scores, _ = score(load_labeled("dev"), extractor)
test_scores, test_errors = score(load_labeled("test"), extractor)

summary = pd.DataFrame({
    "dev F1": dev_scores["f1"],
    "test precision": test_scores["precision"],
    "test recall": test_scores["recall"],
    "test F1": test_scores["f1"],
    "test labeled": test_scores["labeled"].astype(int),
})
for col in ["dev F1", "test precision", "test recall", "test F1"]:
    summary[col] = (summary[col] * 100).round(1).astype(str) + "%"
summary

,dev F1,test precision,test recall,test F1,test labeled
bedrooms,97.7%,96.1%,100.0%,98.0%,98
bathrooms,97.4%,94.6%,95.6%,95.1%,91
sqft,98.7%,97.9%,100.0%,98.9%,47
price,100.0%,100.0%,100.0%,100.0%,2
overall,97.9%,95.9%,98.3%,97.1%,238


## 4. Error analysis

Every test-set error, grouped by why it happened.

In [5]:
PATTERNS = {
    (1171172498, "bedrooms"): ("Count is for part of the home", "2 bedrooms ... on the main level"),
    (1176163450, "bedrooms"): ("Count is for part of the home", "2 bedrooms on each floor"),
    (1173253845, "bedrooms"): ("Count is for part of the home", "1 bedroom features an ensuite bath"),
    (1190768489, "bedrooms"): ("Multi-unit building", "12 residences offering 3 bedrooms"),
    (1190768489, "bathrooms"): ("Multi-unit building", "featuring 2, 2.5 or 3 bathrooms"),
    (1177666601, "bathrooms"): ("Unusual half/fraction wording", "two and 1-half bathrooms"),
    (1183442973, "bathrooms"): ("Unusual half/fraction wording", "four-and-1-half bathrooms"),
    (1173253845, "bathrooms"): ("Unusual half/fraction wording", "a 3/4 bathroom"),
    (1190168632, "bathrooms"): ("Unusual half/fraction wording", "3-quarter bath"),
    (1177295887, "bathrooms"): ("Number word not converted", "two thoughtfully laid-out bathrooms"),
    (1153094861, "bathrooms"): ("Rule too strict", "(optional 3rd bedroom) and 2-bath"),
    (1159195492, "sqft"): ("Lot size named after the number", "3049 square feet, this expansive lot"),
}
errors = test_errors.copy()
errors[["pattern", "example"]] = errors.apply(
    lambda r: pd.Series(PATTERNS.get((r.listing_id, r.entity), ("Other", ""))), axis=1)
errors[["entity", "gold", "predicted", "pattern", "example"]]

,entity,gold,predicted,pattern,example
0,bedrooms,NaN,2.0,Count is for part of the home,2 bedrooms ... on the main level
1,bedrooms,NaN,2.0,Count is for part of the home,2 bedrooms on each floor
2,bedrooms,NaN,3.0,Multi-unit building,12 residences offering 3 bedrooms
3,bedrooms,NaN,1.0,Count is for part of the home,1 bedroom features an ensuite bath
4,bathrooms,2.5,1.0,Unusual half/fraction wording,two and 1-half bathrooms
5,bathrooms,2.0,NaN,Rule too strict,(optional 3rd bedroom) and 2-bath
6,bathrooms,2.0,NaN,Number word not converted,two thoughtfully laid-out bathrooms
7,bathrooms,NaN,3.0,Multi-unit building,"featuring 2, 2.5 or 3 bathrooms"
8,bathrooms,NaN,4.0,Unusual half/fraction wording,a 3/4 bathroom
9,bathrooms,NaN,3.0,Unusual half/fraction wording,3-quarter bath


In [6]:
errors["pattern"].value_counts().rename("errors").to_frame()

,errors
pattern,
Unusual half/fraction wording,4
Count is for part of the home,3
Multi-unit building,2
Rule too strict,1
Number word not converted,1
Lot size named after the number,1


## 5. Amenity check

Amenities come straight from taxonomy matches, so instead of labeling every one, I hand-checked 100 random matches from the test remarks.

In [7]:
spot = pd.read_csv(DATA / "amenity_spot_check.csv")
print(f"Amenity precision: {spot['correct'].mean():.0%} ({spot['correct'].sum()} of {len(spot)} correct)")
spot[~spot["correct"]].groupby("reason")["term"].apply(lambda t: ", ".join(sorted(set(t)))).to_frame("terms")

Amenity precision: 88% (88 of 100 correct)


,terms
reason,
different meaning,"garden, spa"
hypothetical,"garden, parking, tennis courts, workshop"
"nearby, not the property","golf course, spa, walking trails"


## 6. Comparison with the database

Running the extractor on all 1,000 cleaned remarks and comparing with the `beds` and `baths` columns.

In [8]:
df = pd.read_csv(DATA / "listing_sample_clean.csv")
extracted = df["remarks_clean"].apply(extractor.extract_all).apply(pd.Series)

found = extracted[["bedrooms", "bathrooms", "sqft", "price"]].notna().mean()
print("Share of remarks where each entity was found")
print((found * 100).round(1).astype(str) + "%")

both = extracted["bedrooms"].notna() & df["beds"].notna()
beds_match = (extracted.loc[both, "bedrooms"] == df.loc[both, "beds"]).mean()
both = extracted["bathrooms"].notna() & df["baths"].notna()
baths_exact = (extracted.loc[both, "bathrooms"] == df.loc[both, "baths"]).mean()
baths_rounded = (extracted.loc[both, "bathrooms"].apply(math.ceil) == df.loc[both, "baths"]).mean()
print(f"\nBedrooms match the database: {beds_match:.0%}")
print(f"Bathrooms match exactly: {baths_exact:.0%}, after rounding half baths up: {baths_rounded:.0%}")
print(f"Median amenities per remark: {extracted['amenities'].str.len().median():.0f}")

Share of remarks where each entity was found
bedrooms     67.8%
bathrooms    60.1%
sqft         34.3%
price         0.8%
dtype: object

Bedrooms match the database: 87%
Bathrooms match exactly: 65%, after rounding half baths up: 88%
Median amenities per remark: 8


## 7. Key findings

**I. Results**

*The rule-based extractor beats the 85% target on held-out data.*
- Test F1 of 97% overall (bedrooms 98%, bathrooms 95%, square feet 99%)
- Dev and test scores are close (98% vs 97%), so the rules aren't overfit
- Amenity precision is 88% on a hand-checked sample

**II. What goes wrong**

*Most errors come from remarks that describe the home in pieces.*
- Counts for one floor or one room ("2 bedrooms on each floor")
- Multi-unit buildings with several sets of counts
- Unusual bathroom wording ("four-and-1-half", "3-quarter bath")
- Amenities that are nearby or hypothetical ("future tennis court") or mean something else ("garden tub")

**III. What the database comparison shows**

*Text and database fields disagree for real reasons, not just extraction errors.*
- The database rounds half baths up (2.5 is stored as 3)
- Database bedroom counts often include ADUs, guest houses, or lofts
- Bedrooms match 87% and bathrooms match 88% once half baths are rounded

**IV. Next steps**
- Add rules for fraction bathrooms and "and 1-half" wording in the cleaner
- Flag multi-unit listings instead of guessing a single count
- Price is rarely in remarks (under 1%), so query parsing in Week 4 matters more for price